In [1]:
import sys, os
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

Enabled rmm statistics


In [2]:
%LoadCheckpoint /home/colinc/code/dias-benchmarks/notebooks/erikbruin/nlp-on-student-writing-eda/src/annotated_cpu/checkpoints/post_cell_21.pickle

trying: ['FuncFormatter']
me:  0
trying: ['plt']
me:  0
trying: ['glob']
me:  0
trying: ['np']
me:  0
trying: ['pd']
me:  0
trying: ['df']
me:  43
trying: ['sample_submission']
me:  1
trying: ['tqdm']
me:  0
trying: ['text']
me:  43
trying: ['train_txt']
me:  1
trying: ['warnings']
me:  0
trying: ['keys']
me:  43
trying: ['train_first']
me:  19
trying: ['train']
me:  43
trying: ['train_last']
me:  19
trying: ['dt']
me:  43
trying: ['all_gaps']
me:  35
trying: ['train_first_last']
me:  19
trying: ['test_txt']
me:  1
trying: ['stop_english']
me:  43
trying: ['other_words_to_take_out']
me:  43
trying: ['myid']
me:  23
trying: ['total_gaps']
me:  37
trying: ['myword']
me:  23
trying: ['orig_output']
me:  2
trying: ['cols_to_display']
me:  27
trying: ['mylen']
me:  23
trying: ['len_dict']
me:  23
trying: ['IREWR_plug_1']
me:  33
trying: ['counter']
me:  21
trying: ['word_dict']


me:  23
trying: ['i_1']
me:  21
trying: ['add_gap_rows']
me:  39
trying: ['txt_file']
me:  23
trying: ['data']
me:  23
trying: ['t']
me:  23
trying: ['IREWR_tmp2']
me:  33
trying: ['stopwords']
me:  43
trying: ['sample_discourse_id']
me:  3
trying: ['counts_dict']
me:  43
trying: ['ax1']
me:  13
trying: ['i_3']
me:  25
trying: ['ax2']
me:  13
trying: ['last_ones']
me:  25
trying: ['IREWR_plug_2']
me:  31
trying: ['cols_to_merge']
me:  25
trying: ['add_gap_rows_2']
me:  41
trying: ['print_colored_essay']
me:  41
trying: ['df1']
me:  43
trying: ['ax']
me:  15
trying: ['style']
me:  0
trying: ['IREWR_tmp']
me:  31
trying: ['av_per_essay']
me:  15
trying: ['CountVectorizer']
me:  0
Declaring variable FuncFormatter
Declaring variable plt
Declaring variable glob
Declaring variable np
Declaring variable pd
Declaring variable tqdm
Declaring variable warnings
Declaring variable style
Declaring variable CountVectorizer
Declaring variable sample_submission
Declaring variable train_txt
Declaring v

In [3]:
%%RecordEvent
%%time
### cell 22 ###

def get_n_grams(n_grams, top_n=10):
    # Fit once on the entire corpus and transform to a sparse matrix
    texts = train['discourse_text'].astype(str)
    types = train['discourse_type']
    vec = CountVectorizer(
        lowercase=True,
        stop_words='english',
        ngram_range=(n_grams, n_grams)
    )
    X = vec.fit_transform(texts)

    # Build a sparse DataFrame of n-gram counts
    df_counts = pd.DataFrame.sparse.from_spmatrix(
        X,
        index=train.index,
        columns=vec.get_feature_names_out()
    )
    df_counts['discourse_type'] = types.values

    # Group by discourse_type and sum counts in one pass
    grouped = df_counts.groupby('discourse_type').sum()

    # For each discourse_type, pick the top_n n-grams
    records = []
    for dt, row in grouped.iterrows():
        top_grams = row.nlargest(top_n)
        records.extend((dt, gram, int(cnt)) for gram, cnt in top_grams.items())

    return pd.DataFrame(records, columns=['Discourse_type', 'words', 'counts'])

# Compute bigrams
bigrams = get_n_grams(n_grams=2, top_n=10)
bigrams.head()

CPU times: user 15.4 s, sys: 36.8 ms, total: 15.5 s
Wall time: 15.5 s


,Discourse_type,words,counts
0,Claim,texting driving,44
1,Claim,cell phone,34
2,Claim,cell phones,32
3,Claim,phone driving,29
4,Claim,phones driving,20


In [4]:
%Checkpoint /home/colinc/code/dias-benchmarks/notebooks/erikbruin/nlp-on-student-writing-eda/src/rewritten_cpu/o4_mini_high/checkpoints/post_cell_22_try_1.pickle

migration speed (bps): 734560902.2701911
---------------------------
variables to migrate:
sample_discourse_id 32
IREWR_tmp2 954
tqdm 1072
get_n_grams 144
i_1 28
counter 28
add_gap_rows 144
all_gaps 2304
last_ones 106820
cols_to_merge 120
sample_submission 569
test_txt 120
i_3 28
total_gaps 10924
np 72
txt_file 208
IREWR_plug_1 61
other_words_to_take_out 152
orig_output 16
df 15908
ax2 536
stopwords 48
cols_to_display 120
ax1 536
text 4920
warnings 72
IREWR_plug_2 61
style 72
df1 725
keys 120
FuncFormatter 1072
CountVectorizer 1480
counts_dict 5403
add_gap_rows_2 144
plt 72
train_first_last 428
train 806064
glob 144
train_txt 126104
train_first 380
t 166
train_last 643
print_colored_essay 144
pd 72
myid 61
dt 57
myword 28
ax 1364
stop_english 1688
mylen 28
av_per_essay 1841
IREWR_tmp 914
len_dict 589920
word_dict 589920
bigrams 10184
data 2813
---------------------------
variables to recompute:
[]
---------------------------
cells to recompute:
[]
Checkpoint saved to: /home/colinc/code

In [5]:
%PrintCellInfo opt_cell_exec_info

======= Cell 0 =======
Input variables []
Active variables ['train', 'train_txt']
Intermediate variables ['test_txt', 'sample_submission']
Future variables []
Modified dataframes
======= Cell 1 =======
Input variables ['train']
Active variables ['sample_discourse_id']
Intermediate variables []
Future variables ['train', 'train_txt']
Modified dataframes
======= Cell 2 =======
Input variables ['train']
Active variables ['cols_to_display', 'train']
Intermediate variables []
Future variables ['sample_discourse_id', 'train_txt']
Modified dataframes
  train
    Input columns: set()
    Changed columns: set()
    Created columns: {'pred_len', 'discourse_len'}
    Deleted columns: set()
======= Cell 3 =======
Input variables ['cols_to_display', 'train']
Active variables []
Intermediate variables []
Future variables ['cols_to_display', 'sample_discourse_id', 'train_txt', 'train']
Modified dataframes
======= Cell 4 =======
Input variables ['sample_discourse_id', 'train']
Active variables []
Inte

In [6]:

with open("/home/colinc/code/dias-benchmarks/notebooks/erikbruin/nlp-on-student-writing-eda/src/opt_cell_exec_info_22_try_1.pkl", "wb") as f:
    pickle.dump(opt_cell_exec_info[22], f)


In [7]:
opt_output = Out.get(4)

In [ ]:
bigrams_opt = bigrams
%LoadCheckpoint /home/colinc/code/dias-benchmarks/notebooks/erikbruin/nlp-on-student-writing-eda/src/annotated_cpu/checkpoints/post_cell_22.pickle

import numpy as np
import cudf
from elastic.core.common.pandas import is_type_styler
is_orig_output_pd = isinstance(orig_output, (pd.Series, pd.DataFrame, pd.Index))
is_opt_output_pd = isinstance(opt_output, (pd.Series, pd.DataFrame, pd.Index))
is_orig_output_array = isinstance(orig_output, (cudf.pandas._wrappers.numpy.ndarray, np.ndarray))
is_opt_output_array = isinstance(opt_output, (cudf.pandas._wrappers.numpy.ndarray, np.ndarray))
is_orig_output_styler = is_type_styler(type(orig_output))
is_opt_output_styler = is_type_styler(type(opt_output))
if is_orig_output_styler and is_opt_output_styler:
    assert orig_output.to_html() == opt_output.to_html()
elif is_orig_output_styler:
    assert orig_output.to_html() == opt_output.to_html()
elif is_opt_output_styler:
    assert opt_output.to_html() == orig_output

if is_orig_output_pd and is_opt_output_pd:
    assert orig_output.equals(opt_output)
# TODO(jie): this is a hack.
elif ((is_orig_output_pd or is_opt_output_pd) and (is_orig_output_array or is_opt_output_array)) or (is_orig_output_array and is_opt_output_array):
    assert list(orig_output) == list(opt_output)
else:
    assert orig_output == opt_output
